# Data Preparation for Recommender System
This notebook generates a synthetic dataset of historical cart transactions for a pizza chain and performs basic data cleansing. The output tables feed into:
* **01_market_basket_analysis** — FPGrowth-based association rules
* **02_collaborative_filter** — ALS-based user-item recommendations
* **03_two_tower** — Two-tower neural recommender (uses `item_catalog` and `user_features`)

### Compute Requirements
| | |
|---|---|
| **Runtime** | DBR 17.3 ML |
| **Compute** | Classic cluster (single node is sufficient) |

### Output Tables
| Table | Description |
|---|---|
| `cleaned_mapped_dataset` | Full cleaned dataset (all orders) |
| `train_dataset` | 80 % training split (seed=42) |
| `test_dataset` | 20 % held-out test split — shared across all downstream notebooks |
| `item_catalog` | One row per product — category, price, calories, dietary flags |
| `user_features` | One row per user — order stats, preferred category, spend, tier |

In [1]:
catalog = 'users'
schema = 'jon_cheung'

# Output table names
cleaned_table   = f'{catalog}.{schema}.cleaned_mapped_dataset'
train_table     = f'{catalog}.{schema}.train_dataset'
test_table      = f'{catalog}.{schema}.test_dataset'
item_cat_table  = f'{catalog}.{schema}.item_catalog'
user_feat_table = f'{catalog}.{schema}.user_features'

# Ensure the schema exists
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}')

""


In [2]:
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType

random.seed(42)

# ---------------------
# Product catalog
# ---------------------
pizzas = [
    'pepperoni-pizza', 'cheese-pizza', 'supreme-pizza', 'meat-lovers-pizza',
    'veggie-pizza', 'hawaiian-pizza', 'bbq-chicken-pizza', 'deep-dish-pepperoni',
]
sides    = ['crazy-bread', 'italian-cheese-bread', 'caesar-wings', 'stuffed-crazy-bread']
drinks   = ['pepsi', 'mountain-dew', 'diet-pepsi', 'orange-fanta']
desserts = ['cookie-dough-brownie', 'cinnamon-loaded-crazy-bites']

# Items that should NOT be recommended (to be cleaned out)
sauce_list   = ['marinara-sauce', 'ranch-dressing']
utensil_list = ['plastic-fork', 'plastic-knife']

# ---------------------
# Generate transactions
# ---------------------
num_users  = 2_000
num_orders = 15_000
user_ids   = [str(random.randint(10_000_000, 99_999_999)) for _ in range(num_users)]

# Synthetic timestamp window: 2 years ending 2023-12-31
_start      = datetime(2022, 1, 1)
_total_secs = int(timedelta(days=730).total_seconds())

orders = []
for i in range(num_orders):
    mpid     = random.choice(user_ids)
    order_id = f'ORD-{i+1:06d}'

    # Every order has at least 1 pizza
    items = random.sample(pizzas, random.randint(1, 3))

    # Sides (~60 %), drinks (~50 %), desserts (~20 %)
    if random.random() < 0.60:
        items.append(random.choice(sides))
    if random.random() < 0.50:
        items.append(random.choice(drinks))
    if random.random() < 0.20:
        items.append(random.choice(desserts))

    # Sprinkle in items we will later clean out
    if random.random() < 0.35:
        items.append(random.choice(sauce_list))
    if random.random() < 0.25:
        items.append(random.choice(utensil_list))

    order_ts = _start + timedelta(seconds=random.randint(0, _total_secs))
    orders.append((mpid, order_id, ', '.join(items), order_ts.strftime('%Y-%m-%d %H:%M:%S')))

# Build Spark DataFrame (mimics a raw orders table)
raw_schema = StructType([
    StructField('mpid',            StringType()),
    StructField('order_id',        StringType()),
    StructField('order_products',  StringType()),   # comma-separated product names
    StructField('order_timestamp', StringType()),   # synthetic datetime string
])
sdf_raw = spark.createDataFrame(orders, raw_schema)
print(f'Generated {sdf_raw.count():,} raw orders')
display(sdf_raw.limit(10))

Generated 15,000 raw orders


,mpid,order_id,order_products,order_timestamp
0,30513739,ORD-000001,"veggie-pizza, supreme-pizza, stuffed-crazy-bread, orange-fanta, ranch-dressing",2022-12-04 19:52:43
1,19774839,ORD-000002,"pepperoni-pizza, deep-dish-pepperoni, hawaiian-pizza, mountain-dew, ranch-dressing, plastic-fork",2022-07-03 07:16:21
2,68548403,ORD-000003,"hawaiian-pizza, supreme-pizza, meat-lovers-pizza, mountain-dew",2023-08-16 16:55:32
3,93926371,ORD-000004,"supreme-pizza, veggie-pizza, deep-dish-pepperoni, caesar-wings, pepsi, cookie-dough-brownie",2023-03-13 06:20:17
4,23009833,ORD-000005,"veggie-pizza, hawaiian-pizza, caesar-wings, marinara-sauce",2022-12-18 02:41:11
5,82160068,ORD-000006,"deep-dish-pepperoni, pepperoni-pizza, hawaiian-pizza",2022-08-14 12:44:42
6,43554863,ORD-000007,"supreme-pizza, hawaiian-pizza, cinnamon-loaded-crazy-bites, plastic-fork",2023-09-08 05:10:27
7,32097220,ORD-000008,"deep-dish-pepperoni, caesar-wings, mountain-dew, ranch-dressing",2023-07-11 20:36:39
8,26405605,ORD-000009,"bbq-chicken-pizza, deep-dish-pepperoni, crazy-bread, diet-pepsi",2022-09-25 14:13:15
9,66623995,ORD-000010,"supreme-pizza, bbq-chicken-pizza, cheese-pizza, crazy-bread, marinara-sauce",2023-07-05 22:54:33


## Data Cleaning
Remove items we should never recommend — condiment add-ons (sauces) and disposable utensils — then drop any orders that become empty after filtering.

In [3]:
from pyspark.sql.functions import col, expr, to_timestamp, row_number
from pyspark.sql.window import Window

# 1. Split comma-separated string into an array of unique, trimmed product slugs
sdf_prepared = sdf_raw.withColumn(
    'order_product_list',
    expr("array_distinct(transform(split(order_products, ','), x -> trim(x)))")
)

# 2. Define items to remove
sauce_list   = ['marinara-sauce', 'ranch-dressing']
utensil_list = ['plastic-fork', 'plastic-knife']
items_to_remove = sauce_list + utensil_list

# 3. Filter out unwanted items from each order
remove_expr = ', '.join([f"'{item}'" for item in items_to_remove])
sdf_cleaned = sdf_prepared.withColumn(
    'order_product_list',
    expr(f"filter(order_product_list, x -> NOT array_contains(array({remove_expr}), x))")
)

# 4. Drop empty strings and orders with no remaining products
sdf_cleaned = sdf_cleaned.withColumn(
    'order_product_list',
    expr("filter(order_product_list, x -> x != '')")
)

# 5. Cast timestamp string → TimestampType
sdf_cleaned = sdf_cleaned.withColumn('order_timestamp', to_timestamp('order_timestamp'))

# 6. Compute per-user order_number (1 = earliest order, ascending)
w_user_time = Window.partitionBy('mpid').orderBy('order_timestamp')
sdf_cleaned_mapped = (
    sdf_cleaned
    .filter(expr('size(order_product_list) > 0'))
    .withColumn('order_number', row_number().over(w_user_time))
    .select('mpid', 'order_id', 'order_product_list', 'order_timestamp', 'order_number')
)

print(f'Cleaned dataset: {sdf_cleaned_mapped.count():,} orders')
display(sdf_cleaned_mapped.limit(10))

Cleaned dataset: 15,000 orders


,mpid,order_id,order_product_list,order_timestamp,order_number
0,10054484,ORD-008152,"[deep-dish-pepperoni, bbq-chicken-pizza, orange-fanta]",2022-03-15 00:06:20,1
1,10054484,ORD-012542,"[veggie-pizza, stuffed-crazy-bread]",2022-06-14 21:40:27,2
2,10054484,ORD-008148,"[meat-lovers-pizza, hawaiian-pizza, pepperoni-pizza]",2022-10-16 15:34:00,3
3,10054484,ORD-014176,"[deep-dish-pepperoni, cheese-pizza, crazy-bread]",2022-11-16 22:36:54,4
4,10054484,ORD-004960,"[supreme-pizza, deep-dish-pepperoni, pepsi]",2022-12-25 08:03:08,5
5,10054484,ORD-002128,"[veggie-pizza, bbq-chicken-pizza, cheese-pizza, crazy-bread, diet-pepsi]",2023-01-19 06:09:32,6
6,10054484,ORD-009098,[bbq-chicken-pizza],2023-07-15 16:57:16,7
7,10054484,ORD-000800,"[deep-dish-pepperoni, supreme-pizza, bbq-chicken-pizza, caesar-wings, diet-pepsi]",2023-12-02 17:36:30,8
8,10076758,ORD-010500,"[supreme-pizza, cheese-pizza, bbq-chicken-pizza, crazy-bread]",2022-01-01 19:58:20,1
9,10076758,ORD-009392,"[cheese-pizza, stuffed-crazy-bread]",2022-05-03 19:34:39,2


In [4]:
# Save the full cleaned dataset (used by both 01_MBA and 02_collab_filter)
sdf_cleaned_mapped.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(cleaned_table)

# 80/20 train-test split — both downstream notebooks load these directly
# so they evaluate against the exact same held-out set
train, test = sdf_cleaned_mapped.randomSplit([0.8, 0.2], seed=42)

train.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(train_table)

test.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(test_table)

print(f'Saved to {catalog}.{schema}:')
print(f'  {cleaned_table}  {sdf_cleaned_mapped.count():,} orders')
print(f'  {train_table}           {train.count():,} orders')
print(f'  {test_table}            {test.count():,} orders')

Saved to users.jon_cheung:
  users.jon_cheung.cleaned_mapped_dataset  15,000 orders
  users.jon_cheung.train_dataset           12,058 orders
  users.jon_cheung.test_dataset            2,942 orders


## Section 3 — Feature Tables

Generates two supplementary tables used by the two-tower model in notebook 03. The schema of `train_dataset` / `test_dataset` is **not changed** — notebooks 01 and 02 are unaffected.

| Table | Grain | Used by |
|---|---|---|
| `item_catalog` | one row per product slug | item tower features |
| `user_features` | one row per `mpid` | user tower features |

In [5]:
import pandas as pd

# ---------------------
# Item catalog
# All features are deterministic from the product slug or category.
# ---------------------
item_catalog_rows = [
    # slug                        category   price  calories      vegetarian
    ('pepperoni-pizza',            'pizza',   12.99, 'high',       False),
    ('cheese-pizza',               'pizza',   10.99, 'high',       True),
    ('supreme-pizza',              'pizza',   13.99, 'high',       False),
    ('meat-lovers-pizza',          'pizza',   14.99, 'high',       False),
    ('veggie-pizza',               'pizza',   11.99, 'medium',     True),
    ('hawaiian-pizza',             'pizza',   12.99, 'high',       False),
    ('bbq-chicken-pizza',          'pizza',   13.49, 'high',       False),
    ('deep-dish-pepperoni',        'pizza',   15.99, 'high',       False),
    ('crazy-bread',                'side',     4.99, 'medium',     True),
    ('italian-cheese-bread',       'side',     5.99, 'medium',     True),
    ('caesar-wings',               'side',     6.99, 'medium',     False),
    ('stuffed-crazy-bread',        'side',     5.49, 'medium',     True),
    ('pepsi',                      'drink',    2.49, 'low',        True),
    ('mountain-dew',               'drink',    2.49, 'low',        True),
    ('diet-pepsi',                 'drink',    2.49, 'low',        True),
    ('orange-fanta',               'drink',    2.49, 'low',        True),
    ('cookie-dough-brownie',       'dessert',  4.49, 'high',       True),
    ('cinnamon-loaded-crazy-bites','dessert',  3.99, 'medium',     True),
]

item_pdf = pd.DataFrame(
    item_catalog_rows,
    columns=['item', 'category', 'base_price', 'calories_bucket', 'is_vegetarian']
)

item_sdf = spark.createDataFrame(item_pdf)
item_sdf.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(item_cat_table)

print(f'Item catalog saved to {item_cat_table}')
display(item_sdf)

Item catalog saved to users.jon_cheung.item_catalog


,item,category,base_price,calories_bucket,is_vegetarian
0,pepperoni-pizza,pizza,12.99,high,False
1,cheese-pizza,pizza,10.99,high,True
2,supreme-pizza,pizza,13.99,high,False
3,meat-lovers-pizza,pizza,14.99,high,False
4,veggie-pizza,pizza,11.99,medium,True
5,hawaiian-pizza,pizza,12.99,high,False
6,bbq-chicken-pizza,pizza,13.49,high,False
7,deep-dish-pepperoni,pizza,15.99,high,False
8,crazy-bread,side,4.99,medium,True
9,italian-cheese-bread,side,5.99,medium,True


In [6]:
from pyspark.sql.functions import (
    col, explode, count, avg, size, round as spark_round,
    sum as spark_sum, when, max as spark_max, datediff, lit
)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Load the full cleaned dataset and item catalog
orders_sdf = spark.read.table(cleaned_table)
items_sdf  = spark.read.table(item_cat_table).select('item', 'category', 'base_price')

# Explode to one (mpid, item) row per interaction
exploded = (
    orders_sdf
    .withColumn('item', explode(col('order_product_list')))
    .select('mpid', 'order_id', 'order_timestamp', 'item')
)

# Join item prices for spend calculation
exploded_priced = exploded.join(items_sdf, on='item', how='left')

# Order-level aggregates
order_stats = (
    orders_sdf
    .groupBy('mpid')
    .agg(
        count('order_id').alias('total_orders'),
        spark_round(avg(size(col('order_product_list'))), 2).alias('avg_order_size'),
    )
)

# Lifetime spend per user
spend_stats = (
    exploded_priced
    .groupBy('mpid')
    .agg(spark_round(spark_sum('base_price'), 2).alias('lifetime_spend'))
)

# Preferred category: most-ordered category per user
category_counts = (
    exploded_priced
    .groupBy('mpid', 'category')
    .agg(count('*').alias('cat_count'))
)
w_cat = Window.partitionBy('mpid').orderBy(col('cat_count').desc())
preferred_category = (
    category_counts
    .withColumn('rn', row_number().over(w_cat))
    .filter(col('rn') == 1)
    .select('mpid', col('category').alias('preferred_category'))
)

# Recency: days since each user's most recent order
# Reference date = day after the latest order in the dataset (simulates 'today')
ref_ts = orders_sdf.agg(spark_max('order_timestamp').alias('ref')).collect()[0]['ref']
recency_stats = (
    orders_sdf
    .groupBy('mpid')
    .agg(spark_max('order_timestamp').alias('last_order_ts'))
    .withColumn(
        'days_since_last_order',
        datediff(lit(ref_ts).cast('timestamp'), col('last_order_ts'))
    )
)

# Assemble all features
user_features = (
    order_stats
    .join(spend_stats,         on='mpid', how='left')
    .join(preferred_category,  on='mpid', how='left')
    .join(recency_stats,       on='mpid', how='left')
    .withColumn(
        'customer_tier',
        when(col('total_orders') >= 8, 'Gold')
        .when(col('total_orders') >= 4, 'Silver')
        .otherwise('Bronze')
    )
    .withColumn(
        'recency_bucket',
        when(col('days_since_last_order') <= 30,  'Active')
        .when(col('days_since_last_order') <= 90, 'Lapsed')
        .otherwise('Churned')
    )
)

user_features.write.format('delta').mode('overwrite') \
    .option('overwriteSchema', 'true').saveAsTable(user_feat_table)

print(f'User features saved to {user_feat_table}  ({user_features.count():,} users)')
display(user_features.orderBy(col('total_orders').desc()).limit(10))

User features saved to users.jon_cheung.user_features  (1,998 users)


,mpid,total_orders,avg_order_size,lifetime_spend,preferred_category,last_order_ts,days_since_last_order,customer_tier,recency_bucket
0,19584466,17,3.12,516.47,pizza,2023-12-29 06:56:27,2,Gold,Active
1,77005685,17,3.47,544.91,pizza,2023-12-03 10:44:14,28,Gold,Active
2,58970845,16,3.88,593.88,pizza,2023-12-07 01:57:37,24,Gold,Active
3,62274692,16,3.50,539.94,pizza,2023-12-22 18:44:14,9,Gold,Active
4,87267850,16,3.50,560.44,pizza,2023-12-25 19:45:24,6,Gold,Active
5,23756669,15,3.13,459.03,pizza,2023-11-30 04:56:10,31,Gold,Lapsed
6,79244185,15,3.60,546.96,pizza,2023-12-26 01:34:18,5,Gold,Active
7,17148018,15,3.73,536.44,pizza,2023-11-17 19:33:56,44,Gold,Lapsed
8,98010774,15,3.53,519.97,pizza,2023-10-01 14:29:09,91,Gold,Churned
9,77349752,15,3.13,451.53,pizza,2023-10-31 10:48:50,61,Gold,Lapsed
